# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [ ]:
import sys
from pathlib import Path

for _root in (Path.cwd(), Path.cwd() / 'mp1-prompt-lab'):
    if (_root / 'mp1_prompt_lab.py').exists():
        sys.path.insert(0, str(_root.resolve()))
        break

import mp1_prompt_lab as lab
import pandas as pd

lab.load_env()
client = lab.make_client()
print('Setup complete. Helpers loaded from mp1_prompt_lab.py')

## Step 1 — Load the data

In [ ]:
snippets = lab.load_snippets()
golden = lab.load_golden()
print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [ ]:
from mp1_prompt_lab import (
    STRATEGIES,
    prompt_cot,
    prompt_few_shot,
    prompt_structured,
    prompt_zero_shot,
)

# Four strategies live in mp1_prompt_lab.py (zero-shot, few-shot, structured, CoT).
print('Strategies:', list(STRATEGIES))
print(prompt_zero_shot(snippets[0]['snippet'])[0]['content'][:120], '...')

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [ ]:
from mp1_prompt_lab import parse_response, run_all, run_one

print('parse_response demo:', parse_response('```json\n{"company": "Acme Corp"}\n```'))

In [ ]:
import pickle

CACHE_PATH = lab.RESULTS_CACHE

if CACHE_PATH.exists():
    results = pickle.loads(CACHE_PATH.read_bytes())
    print(f'Loaded {len(results)} cached extraction results from {CACHE_PATH.name}.')
else:
    results = await lab.run_all(client, snippets)
    CACHE_PATH.write_bytes(pickle.dumps(results))
    print(f'Got {len(results)} results and wrote {CACHE_PATH.name}.')

print(f'Total extraction cost: ${sum(r["cost_usd"] for r in results):.6f}')
results[0]

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [ ]:
from mp1_prompt_lab import score_accuracy, score_llm_judge, _norm_str, _norm_years

In [ ]:
# Offline checks for parse + accuracy — no API calls
lab.run_offline_checks()

In [ ]:
import asyncio
import pickle

SCORED_CACHE = lab.SCORED_CACHE
snippet_by_id = {row['id']: row['snippet'] for row in snippets}

if SCORED_CACHE.exists():
    scored = pickle.loads(SCORED_CACHE.read_bytes())
    print(f'Loaded {len(scored)} cached scored rows from {SCORED_CACHE.name}.')
else:
    scored = list(
        await asyncio.gather(
            *[lab.score_row(client, row, golden, snippet_by_id) for row in results]
        )
    )
    SCORED_CACHE.write_bytes(pickle.dumps(scored))
    print(f'Scored {len(scored)} results and wrote {SCORED_CACHE.name}.')

## Step 5 — Build the comparison table

In [ ]:
summary, by_field = lab.comparison_tables(scored, golden)
print(summary.to_markdown())
print()
print('By-field hit rate')
print(by_field.to_markdown())
summary

## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```